# Learnable PPR — Planetoid (Cora / CiteSeer / PubMed)

**Phase 1**: Architecture search learns per-edge `(teleport_u, teleport_v)` via bi-level optimization.\
**Phase 2**: Fine-tune a GNN on the extracted PPR subgraphs.

Uses real PyG node features. Inspired by [PS2 (WSDM'23)](https://github.com/qiaoyu-tan/PS2).

In [ ]:
TELEPORT_VALUES = [0.50, 0.85, 0.95]
ALPHA      = 0.5
PPR_EPSILON = 1e-3
PPR_WINDOW  = 10

DATASETS = ['Cora', 'CiteSeer', 'PubMed']
ENCODERS = ['GCN', 'SAGE', 'GAT']

# Architecture (feature_dim is read per-dataset from real node features)
HIDDEN_CHANNELS = 256
NUM_LAYERS  = 3
DROPOUT     = 0.3

# Phase 1: Architecture Search
SEARCH_EPOCHS          = 50
SEARCH_BATCH_SIZE      = 256
SEARCH_LR              = 0.01
TEMPERATURE_START      = 1.0
TEMPERATURE_END        = 0.2
EDGES_PER_SEARCH_EPOCH = None   # None = all (small graphs)
FIRST_ORDER   = True
SEARCH_VAL_EVERY = 5
USE_AMP = True

# Phase 2: Fine-tuning
FINETUNE_EPOCHS    = 500
FINETUNE_BATCH_SIZE = 512
FINETUNE_LR        = 0.005
FINETUNE_PATIENCE  = 30
MAX_SUBGRAPHS_PER_FORWARD = 256
EDGES_PER_EPOCH = None   # None = all

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Search space: {len(TELEPORT_VALUES)}x{len(TELEPORT_VALUES)} = {len(TELEPORT_VALUES)**2} configs')
print(f'Datasets: {DATASETS}  |  Encoders: {ENCODERS}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
import json, os, time, copy
import pandas as pd

SEED = 42
import torch
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
import random
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

sns.set_theme(style='whitegrid', palette='muted')

from subgrapher.benchmark.data_prep import prepare_planetoid_data
from subgrapher.utils.models import GCN, SAGE, GAT, LinkPredictor
from subgrapher.benchmark_learnable_ppr.multi_scale_ppr import MultiScalePPR
from subgrapher.benchmark_learnable_ppr.autolink_ppr import AutoLinkPPR
from subgrapher.benchmark_learnable_ppr.search_net import PPRSearchNet
from subgrapher.benchmark_learnable_ppr.searcher import ArchitectureSearcher
from subgrapher.benchmark_learnable_ppr.finetuner import (
    finetune_on_subgraphs, LearnablePPRExtractor, build_or_load_cache)
from subgrapher.benchmark_learnable_ppr.evaluator import (
    evaluate_learnable_ppr, evaluate_learnable_ppr_fullgraph, print_evaluation_results)
from subgrapher.benchmark_learnable_ppr.artifacts import save_learnable_ppr_experiment


def ensure_train_only_edges(data, dd):
    if not getattr(data, '_edge_index_train_only', False):
        data._orig_edge_index = data.edge_index
        data.edge_index = dd['train_edge_index']
        data._edge_index_train_only = True
        print(f'  [train-only edges] swapped: '
              f"{data._orig_edge_index.size(1):,} -> {data.edge_index.size(1):,}")
    return data

def _make_encoder(enc_type, in_ch, hid=HIDDEN_CHANNELS,
                  n_layers=NUM_LAYERS, dropout=DROPOUT):
    if enc_type == 'GCN':  return GCN(in_ch, hid, hid, n_layers, dropout)
    if enc_type == 'SAGE': return SAGE(in_ch, hid, hid, n_layers, dropout)
    if enc_type == 'GAT':  return GAT(in_ch, hid, hid, n_layers, dropout, heads=4)
    raise ValueError(enc_type)

## 1. Data Loading and Multi-Scale PPR

In [ ]:
all_results = {}

for dataset_name in DATASETS:
    print(f'\n{"=" * 60}')
    print(f'Dataset: {dataset_name}')
    print(f'{"=" * 60}')

    dd = prepare_planetoid_data(dataset_name)
    data = dd['data']
    split_edge = dd['split_edge']
    feature_dim = dd['feature_dim']

    print(f'Nodes: {data.num_nodes:,}  Edges: {data.edge_index.size(1):,}  '
          f'Feature dim: {feature_dim}')
    print(f'Train: {split_edge["train"]["source_node"].size(0):,}  '
          f'Val: {split_edge["valid"]["source_node"].size(0):,}  '
          f'Test: {split_edge["test"]["source_node"].size(0):,}')

    ensure_train_only_edges(data, dd)

    print(f'\nLoading multi-scale PPR (teleport={TELEPORT_VALUES})...')
    multi_scale_ppr = MultiScalePPR(
        dataset_name, data, teleport_values=TELEPORT_VALUES, device=DEVICE)
    print(multi_scale_ppr)

    all_results[dataset_name] = {
        'data': data, 'split_edge': split_edge,
        'dd': dd, 'feature_dim': feature_dim,
        'multi_scale_ppr': multi_scale_ppr, 'experiments': {},
    }

## 2. Phase 1: Architecture Search

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    data = r['data']; split_edge = r['split_edge']; msp = r['multi_scale_ppr']
    feature_dim = r['feature_dim']
    ensure_train_only_edges(data, r['dd'])

    for enc_type in ENCODERS:
        print(f'\n{"=" * 60}')
        print(f'Phase 1: {dataset_name} / {enc_type}  (in_ch={feature_dim})')
        print(f'{"=" * 60}')

        model = AutoLinkPPR(
            in_channels=feature_dim, hidden_channels=HIDDEN_CHANNELS,
            num_layers=NUM_LAYERS, dropout=DROPOUT,
            gnn_type=enc_type, num_configs=msp.num_configs)

        arch_net = PPRSearchNet(
            in_channels=HIDDEN_CHANNELS, hidden_channels=256,
            num_layers=3, temperature=TEMPERATURE_START)

        searcher = ArchitectureSearcher(
            model, arch_net, msp, data, split_edge,
            device=DEVICE, lr=SEARCH_LR, lr_arch=SEARCH_LR,
            temperature_start=TEMPERATURE_START,
            temperature_end=TEMPERATURE_END,
            edges_per_search_epoch=EDGES_PER_SEARCH_EPOCH,
            first_order=FIRST_ORDER)

        search_history = searcher.search(
            epochs=SEARCH_EPOCHS, batch_size=SEARCH_BATCH_SIZE,
            verbose=True, val_every=SEARCH_VAL_EVERY, use_amp=USE_AMP)

        print(f'Search done in {search_history["total_time"]:.1f}s')
        r['experiments'][enc_type] = {
            'search_history': search_history,
            'model': model, 'arch_net': arch_net,
        }

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    data = r['data']; split_edge = r['split_edge']; msp = r['multi_scale_ppr']
    ensure_train_only_edges(data, r['dd'])

    for enc_type in ENCODERS:
        exp = r['experiments'][enc_type]
        searcher_ref = ArchitectureSearcher(
            exp['model'], exp['arch_net'], msp, data, split_edge,
            device=DEVICE, first_order=FIRST_ORDER)

        print(f'Extracting configs: {dataset_name} / {enc_type}')
        train_configs, train_counts = searcher_ref.get_edge_configs('train')
        val_configs,   val_counts   = searcher_ref.get_edge_configs('valid')
        test_configs,  test_counts  = searcher_ref.get_edge_configs('test')

        exp.update({
            'train_configs': train_configs, 'val_configs': val_configs,
            'test_configs': test_configs,
            'train_counts': train_counts, 'val_counts': val_counts,
            'test_counts': test_counts,
        })
        print(f'  Train config counts: {train_counts.tolist()}')

        p1_dir = f'results/phase1-checkpoint/{dataset_name}/{enc_type}'
        os.makedirs(p1_dir, exist_ok=True)
        sh = exp['search_history']
        with open(os.path.join(p1_dir, 'search_history.json'), 'w') as f:
            json.dump({
                'search_loss': sh['search_loss'],
                'val_loss': [v for v in sh['val_loss'] if v is not None],
                'best_val_loss': sh['best_val_loss'],
                'best_epoch': sh['best_epoch'],
                'total_time': sh['total_time'],
                'temperature': sh['temperature'],
                'arch_entropy': sh['arch_entropy'],
                'softmax_mass_top1': sh['softmax_mass_top1'],
                'config_distributions': sh['config_distributions'],
                'train_counts': train_counts.tolist(),
                'val_counts': val_counts.tolist(),
                'test_counts': test_counts.tolist(),
            }, f, indent=2)
        torch.save({
            'model_state': exp['model'].state_dict(),
            'arch_net_state': exp['arch_net'].state_dict(),
            'train_configs': train_configs,
            'val_configs': val_configs,
            'test_configs': test_configs,
        }, os.path.join(p1_dir, 'phase1_checkpoint.pt'))
        print(f'  Checkpoint saved: {p1_dir}')

## 3. Config Distribution Heatmap

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    msp = r['multi_scale_ppr']; S = msp.num_scales

    for enc_type in ENCODERS:
        exp = r['experiments'][enc_type]
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        for ax, (split, counts) in zip(axes, [
            ('Train', exp['train_counts']),
            ('Valid', exp['val_counts']),
            ('Test',  exp['test_counts']),
        ]):
            matrix = counts.numpy().reshape(S, S).astype(float)
            total = matrix.sum()
            pct = 100 * matrix / max(total, 1)
            annot = np.array([[f'{int(matrix[i,j])}\n({pct[i,j]:.1f}%)'
                               for j in range(S)] for i in range(S)])
            sns.heatmap(pct, annot=annot, fmt='',
                        xticklabels=[f'{t}' for t in msp.teleport_values],
                        yticklabels=[f'{t}' for t in msp.teleport_values],
                        cmap='YlOrRd', vmin=0, ax=ax)
            ax.set_xlabel('teleport_v'); ax.set_ylabel('teleport_u')
            ax.set_title(f'{split} ({int(total)} edges)')
        fig.suptitle(f'{dataset_name} / {enc_type} -- Learned Config Distribution',
                     fontsize=14, fontweight='bold')
        plt.tight_layout(); plt.show()

## 4. Phase 1 Diagnostic Dashboard

In [ ]:
import math

for dataset_name in DATASETS:
    r = all_results[dataset_name]
    for enc_type in ENCODERS:
        hist = r['experiments'][enc_type]['search_history']
        if not hist.get('temperature'): continue
        n_ep = len(hist['temperature'])
        epochs_list = list(range(1, n_ep + 1))

        fig, axes = plt.subplots(2, 2, figsize=(16, 10))

        ax = axes[0, 0]
        losses = hist['search_loss']
        nan_mask = [math.isnan(v) if isinstance(v, float) else False for v in losses]
        clean_losses = [v if not m else None for v, m in zip(losses, nan_mask)]
        ax.plot(epochs_list, clean_losses, label='Train Loss')
        if any(nan_mask):
            for ne in [e for e, m in zip(epochs_list, nan_mask) if m]:
                ax.axvspan(ne - 0.5, ne + 0.5, color='red', alpha=0.15)
        val_epochs = [e for e, v in zip(epochs_list, hist['val_loss']) if v is not None]
        val_vals   = [v for v in hist['val_loss'] if v is not None]
        if val_vals:
            ax.plot(val_epochs, val_vals, marker='.', label='Val Loss')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        ax.set_title('Search Loss'); ax.legend()

        ax1 = axes[0, 1]
        ax1.plot(epochs_list, hist['temperature'], color='tab:blue', label='Temperature')
        ax1.set_ylabel('Temperature', color='tab:blue')
        ax2 = ax1.twinx()
        ent_clean = [v if not (isinstance(v, float) and math.isnan(v)) else None
                     for v in hist['arch_entropy']]
        ax2.plot(epochs_list, ent_clean, color='tab:orange', label='Entropy')
        if hist.get('softmax_mass_top1'):
            mass_clean = [v if not (isinstance(v, float) and math.isnan(v)) else None
                          for v in hist['softmax_mass_top1']]
            ax2.plot(epochs_list, mass_clean, color='tab:green', linestyle='--', label='Top-1 mass')
        lines1, lab1 = ax1.get_legend_handles_labels()
        lines2, lab2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, lab1 + lab2, loc='center right')
        ax1.set_title('Temperature + Entropy')

        ax = axes[1, 0]
        cdists = hist.get('config_distributions', [])
        if cdists:
            msp = r['multi_scale_ppr']
            labels = [f'({tu},{tv})' for tu, tv in msp.config_labels]
            snap_epochs = [d['epoch'] for d in cdists]
            cmat = np.array([d['train_counts'] for d in cdists], dtype=float)
            pcts = 100 * cmat / cmat.sum(axis=1, keepdims=True).clip(min=1)
            bottom = np.zeros(len(snap_epochs))
            for ci in range(pcts.shape[1]):
                ax.bar(snap_epochs, pcts[:, ci], bottom=bottom, label=labels[ci], width=2)
                bottom += pcts[:, ci]
            ax.legend(fontsize=7, ncol=3, loc='upper right')
        ax.set_title('Config Distribution Over Time')

        ax = axes[1, 1]
        exp = r['experiments'][enc_type]
        total_train = exp['train_counts'].sum().item()
        max_pct = 100 * exp['train_counts'].max().item() / max(total_train, 1)
        dom_idx = exp['train_counts'].argmax().item()
        msp = r['multi_scale_ppr']
        dom_label = f'({msp.config_labels[dom_idx][0]},{msp.config_labels[dom_idx][1]})'
        status_lines = [
            f'Dominant config: {dom_label} ({max_pct:.1f}%)',
            f'Total time: {hist["total_time"]:.0f}s',
            f'Best val loss: {hist["best_val_loss"]:.4f}',
            f'Best epoch: {hist["best_epoch"]}',
        ]
        if max_pct > 95:
            status_lines.insert(0, 'WARNING: CONFIG COLLAPSE (>95%)')
            ax.set_facecolor('#fff0f0')
        else:
            status_lines.insert(0, 'Status: OK')
            ax.set_facecolor('#f0fff0')
        ax.text(0.5, 0.5, '\n'.join(status_lines),
                transform=ax.transAxes, ha='center', va='center',
                fontsize=12, fontfamily='monospace')
        ax.set_title('Phase 1 Summary'); ax.axis('off')

        fig.suptitle(f'{dataset_name} / {enc_type} -- Phase 1 Diagnostics',
                     fontsize=15, fontweight='bold')
        plt.tight_layout(); plt.show()

## 5. Phase 2: Fine-tune on Extracted Subgraphs

### 5a. Build / load subgraph caches (one-time cost)

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    data = r['data']; split_edge = r['split_edge']; msp = r['multi_scale_ppr']
    feature_dim = r['feature_dim']
    ensure_train_only_edges(data, r['dd'])

    for enc_type in ENCODERS:
        exp = r['experiments'][enc_type]
        exp['ft_encoder']  = _make_encoder(enc_type, in_ch=feature_dim)
        exp['ft_predictor'] = LinkPredictor(
            HIDDEN_CHANNELS, HIDDEN_CHANNELS, 1, NUM_LAYERS, DROPOUT)
        exp['ckpt_dir']  = f'checkpoints/learnable-ppr/{dataset_name}/{enc_type}'
        exp['cache_dir'] = f'cache/learnable-ppr/{dataset_name}/{enc_type}'

        print(f'\n[Cache] {dataset_name} / {enc_type}')
        extractor = LearnablePPRExtractor(
            data, msp, exp['train_configs'],
            alpha=ALPHA, epsilon=PPR_EPSILON, window=PPR_WINDOW)
        build_or_load_cache(extractor, split_edge, 'train', exp['cache_dir'])

print('Caches ready.')

### 5b. Fine-tune (resumes from checkpoint if interrupted)

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    data = r['data']; split_edge = r['split_edge']; msp = r['multi_scale_ppr']
    ensure_train_only_edges(data, r['dd'])

    for enc_type in ENCODERS:
        print(f'\nPhase 2: {dataset_name} / {enc_type}')
        exp = r['experiments'][enc_type]

        ft_history = finetune_on_subgraphs(
            exp['ft_encoder'], exp['ft_predictor'], data, split_edge,
            msp, exp['train_configs'],
            alpha=ALPHA, epsilon=PPR_EPSILON, window=PPR_WINDOW,
            epochs=FINETUNE_EPOCHS, batch_size=FINETUNE_BATCH_SIZE,
            lr=FINETUNE_LR, eval_steps=5, device=DEVICE,
            verbose=True, patience=FINETUNE_PATIENCE,
            max_subgraphs_per_forward=MAX_SUBGRAPHS_PER_FORWARD,
            checkpoint_dir=exp['ckpt_dir'],
            cache_dir=exp['cache_dir'],
            edges_per_epoch=EDGES_PER_EPOCH,
            val_config_indices=exp['val_configs'])

        exp['ft_history'] = ft_history
        print(f'Best MRR: {ft_history["best_val_mrr"]:.4f} at epoch {ft_history["best_epoch"]}')

## 6. Test Evaluation

- **Full-graph eval**: scores all node pairs using full-graph embeddings. Comparable to Full Graph baseline.
- **Subgraph eval**: per-edge subgraph scoring (matches training distribution, reference only).

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    data = r['data']; split_edge = r['split_edge']; msp = r['multi_scale_ppr']
    ensure_train_only_edges(data, r['dd'])

    for enc_type in ENCODERS:
        exp = r['experiments'][enc_type]
        cache_dir = exp['cache_dir']

        test_fg = evaluate_learnable_ppr_fullgraph(
            exp['ft_encoder'], exp['ft_predictor'],
            data, split_edge, split='test', device=DEVICE)
        exp['test_results_fullgraph'] = test_fg
        exp['test_results'] = test_fg

        test_sub = evaluate_learnable_ppr(
            exp['ft_encoder'], exp['ft_predictor'],
            data, split_edge, msp, exp['test_configs'],
            split='test', alpha=ALPHA, epsilon=PPR_EPSILON,
            window=PPR_WINDOW, device=DEVICE, cache_dir=cache_dir)
        exp['test_results_subgraph'] = test_sub

        print(f'\n{dataset_name} / {enc_type}')
        print('--- Full-graph eval (comparable to baselines) ---')
        print_evaluation_results(test_fg, 'test')
        print('--- Subgraph eval (reference) ---')
        print_evaluation_results(test_sub, 'test (subgraph)')

## 7. Fine-Tuning Training Curves

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    for enc_type in ENCODERS:
        exp = r['experiments'][enc_type]
        hist = exp.get('ft_history', {})
        if not hist: continue

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        epochs_list = range(1, len(hist['train_loss']) + 1)
        sns.lineplot(x=list(epochs_list), y=hist['train_loss'],
                     ax=axes[0], label='Train Loss')
        axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
        axes[0].set_title('Phase 2: Fine-tune Loss')

        if hist['val_results']:
            val_mrrs = [vr['mrr'] for vr in hist['val_results']]
            eval_epochs = list(range(5, 5 * len(val_mrrs) + 1, 5))
            sns.lineplot(x=eval_epochs, y=val_mrrs,
                         ax=axes[1], marker='o', label='MRR')
            axes[1].axhline(hist['best_val_mrr'], color='red', linestyle='--',
                            alpha=0.5, label=f'Best: {hist["best_val_mrr"]:.4f}')
            axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MRR')
            axes[1].set_title('Validation MRR'); axes[1].legend()

            for metric, label in [('auc', 'AUC'), ('ap', 'AP')]:
                vals = [vr.get(metric, 0) for vr in hist['val_results']]
                if any(vals):
                    sns.lineplot(x=eval_epochs, y=vals, ax=axes[2], label=label)
            axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Score')
            axes[2].set_title('Val AUC / AP'); axes[2].legend()

        fig.suptitle(f'{dataset_name} / {enc_type} -- Fine-tuning',
                     fontsize=14, fontweight='bold')
        plt.tight_layout(); plt.show()

## 8. Cross-Method Comparison

In [ ]:
def load_static_results(base_dir, ds_filter=None, enc_filter=None):
    rows = []
    if not os.path.exists(base_dir): return rows
    for ds in os.listdir(base_dir):
        if ds_filter and ds not in ds_filter: continue
        for exp_name in os.listdir(os.path.join(base_dir, ds)):
            rp = os.path.join(base_dir, ds, exp_name, 'full_results.json')
            if os.path.exists(rp):
                with open(rp) as f:
                    rows.append(json.load(f))
    return rows

comparison_rows = []

# Full Graph baselines
for b in load_static_results('results/benchmark', DATASETS):
    if b['encoder'] in ENCODERS:
        comparison_rows.append({
            'Dataset': b['dataset'], 'Encoder': b['encoder'],
            'Method': 'Full Graph',
            'MRR': b['test_results']['mrr'],
            'AUC': b['test_results']['auc'],
        })

# Static PPR baselines
for b in load_static_results('results/benchmark-ppr', DATASETS):
    if b['encoder'] in ENCODERS:
        lbl = f'Static PPR (a={b.get("ppr_alpha","?")}, e={b.get("ppr_epsilon","?")})'
        comparison_rows.append({
            'Dataset': b['dataset'], 'Encoder': b['encoder'],
            'Method': lbl,
            'MRR': b['test_results']['mrr'],
            'AUC': b['test_results']['auc'],
        })

# Static k-hop baselines
for b in load_static_results('results/benchmark-khop', DATASETS):
    if b['encoder'] in ENCODERS:
        comparison_rows.append({
            'Dataset': b['dataset'], 'Encoder': b['encoder'],
            'Method': f'Static k-hop (k={b.get("k","?")})',
            'MRR': b['test_results']['mrr'],
            'AUC': b['test_results']['auc'],
        })

# Learnable PPR (this notebook)
for ds in DATASETS:
    for enc in ENCODERS:
        tr = all_results.get(ds, {}).get('experiments', {}).get(enc, {}).get('test_results')
        if tr:
            comparison_rows.append({
                'Dataset': ds, 'Encoder': enc, 'Method': 'Learnable PPR',
                'MRR': tr['mrr'], 'AUC': tr['auc'],
            })

if comparison_rows:
    df = pd.DataFrame(comparison_rows)
    for metric in ['MRR', 'AUC']:
        fig, axes = plt.subplots(1, len(DATASETS), figsize=(6 * len(DATASETS), 5),
                                  sharey=False)
        if len(DATASETS) == 1: axes = [axes]
        for ax, ds in zip(axes, DATASETS):
            sub = df[df['Dataset'] == ds]
            sns.barplot(data=sub, x='Encoder', y=metric, hue='Method', ax=ax)
            ax.set_title(ds); ax.set_xlabel('')
            ax.legend(loc='lower right', fontsize=8)
        fig.suptitle(f'Test {metric} — All Methods', fontsize=14, fontweight='bold')
        plt.tight_layout(); plt.show()
    display(df.pivot_table(index=['Dataset', 'Encoder'], columns='Method',
                           values=['MRR', 'AUC']).round(4))
else:
    print('No comparison data yet — run benchmark_runner_planetoid.ipynb first.')

## 9. Per-Edge Config by Node Degree

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    data = r['data']; msp = r['multi_scale_ppr']

    degrees = torch.zeros(data.num_nodes, dtype=torch.long)
    row = data.edge_index[0]
    for i in range(row.size(0)):
        degrees[row[i]] += 1

    for enc_type in ENCODERS:
        exp = r['experiments'][enc_type]
        train_src = r['split_edge']['train']['source_node']
        configs = exp['train_configs']
        src_degrees = degrees[train_src].numpy()
        config_labels = [f'({tu},{tv})' for tu, tv in msp.config_labels]
        config_names  = np.array([config_labels[c.item()] for c in configs])

        degree_bins = np.digitize(src_degrees, bins=[0, 5, 10, 20, 50])
        bin_labels  = ['1-5', '6-10', '11-20', '21-50', '50+']
        df_deg = pd.DataFrame({
            'Degree Bin': [bin_labels[min(b-1, len(bin_labels)-1)] for b in degree_bins],
            'Config': config_names,
        })
        fig, ax = plt.subplots(figsize=(12, 5))
        sns.countplot(data=df_deg, x='Degree Bin', hue='Config', order=bin_labels, ax=ax)
        ax.set_title(f'{dataset_name} / {enc_type} -- Config by Source Degree')
        ax.legend(title='(teleport_u, teleport_v)',
                  bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout(); plt.show()

## 10. Save Results

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]; msp = r['multi_scale_ppr']
    feature_dim = r['feature_dim']

    for enc_type in ENCODERS:
        exp = r['experiments'][enc_type]
        tr = exp.get('test_results')
        if not tr: continue

        save_dir = f'results/benchmark-learnable-ppr/{dataset_name}/{enc_type}'
        os.makedirs(save_dir, exist_ok=True)

        tr_fg  = exp.get('test_results_fullgraph', tr)
        tr_sub = exp.get('test_results_subgraph', {})
        sh = exp['search_history']
        fh = exp['ft_history']
        run_id = time.strftime('%Y%m%d_%H%M%S')

        result_json = {
            'dataset': dataset_name, 'encoder': enc_type,
            'teleport_values': TELEPORT_VALUES, 'alpha': ALPHA,
            'ppr_epsilon': PPR_EPSILON, 'ppr_window': PPR_WINDOW,
            'test_results': {k: float(v) for k, v in tr_fg.items()},
            'test_results_subgraph': {k: float(v) for k, v in tr_sub.items()} if tr_sub else {},
            'test_results_fullgraph': {k: float(v) for k, v in tr_fg.items()},
            'eval_mode_for_comparison': 'fullgraph',
            'config_distribution': {
                'train': exp['train_counts'].tolist(),
                'val':   exp['val_counts'].tolist(),
                'test':  exp['test_counts'].tolist(),
                'labels': [f'({tu},{tv})' for tu, tv in msp.config_labels],
            },
            'search_time':    sh['total_time'],
            'finetune_time':  fh.get('total_time', 0),
            'train_time':     sh['total_time'] + fh.get('total_time', 0),
            'finetune_best_mrr':   fh['best_val_mrr'],
            'finetune_best_epoch': fh['best_epoch'],
            'temperature_start': TEMPERATURE_START,
            'temperature_end':   TEMPERATURE_END,
            'config': {
                'feature_dim': feature_dim,
                'hidden_channels': HIDDEN_CHANNELS,
                'num_layers': NUM_LAYERS, 'dropout': DROPOUT,
                'search_epochs': SEARCH_EPOCHS,
                'finetune_epochs': FINETUNE_EPOCHS,
                'finetune_lr': FINETUNE_LR,
                'finetune_patience': FINETUNE_PATIENCE,
            },
            'seed': SEED,
            'run_id': run_id,
            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        }

        run_dir = os.path.join(save_dir, 'runs', run_id)
        os.makedirs(run_dir, exist_ok=True)
        for p in [os.path.join(run_dir, 'full_results.json'),
                  os.path.join(save_dir, 'full_results.json')]:
            with open(p, 'w') as f:
                json.dump(result_json, f, indent=2)
        print(f'Saved: {run_dir}/full_results.json')
        print(f'Saved: {save_dir}/full_results.json (latest)')

        alpha_save = ALPHA if isinstance(ALPHA, list) else [float(ALPHA)]
        save_learnable_ppr_experiment(
            run_dir, dataset_name, enc_type, run_id,
            TELEPORT_VALUES, alpha_save, PPR_EPSILON, exp, msp,
            extra_config=result_json['config'])

print('Done!')